# Session 1: packets and hops
**A. Real-world problem.** When you open a web page, the server does not send it as one
piece. The page is cut into many small *packets*. Each packet is labelled with where it is
going, and routers pass it on from one to the next, like post offices passing a parcel. The
packets may even arrive in the wrong order, so they carry a number.

**Plan.** (1) cut a message into numbered packets and put it back together, (2) follow one
route hop by hop, (3) count how many time steps a whole message needs, (4) see why very
small and very large packets are both slow.

In [ ]:
import math
import pathlib
import random

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

HEADER_BYTES = 40        # label added to every packet (a typical size for the address labels)
LINK_RATE = 1_000_000    # bits per second on every link in this session (slow, to see the effect)

## Step 1: cut a message into numbered packets
A packet is a piece of the message (the *payload*) plus a sequence number. The number lets
the receiver sort the pieces even if they arrive in a different order.

In [ ]:
def packetize(message, size):
    """Cut `message` (bytes) into [(sequence number, payload)] with payloads of at most `size` bytes."""
    if size <= 0:
        raise ValueError("packet size must be positive")
    return [(i, message[k:k + size]) for i, k in enumerate(range(0, len(message), size))]


def reassemble(packets):
    """Put packets back in order of their sequence number and join the payloads."""
    return b"".join(data for _, data in sorted(packets))


def packet_count(message_bytes, size):
    """How many packets a message of `message_bytes` needs: round up."""
    return math.ceil(message_bytes / size)

## Step 2: routers pass packets hop by hop
Each router knows only the *next hop* for each destination. No router knows the whole path.
Following the next hops one after another gives the route.

In [ ]:
NEXT_HOP = {
    "laptop": {"server": "home router"},
    "home router": {"server": "ISP router"},
    "ISP router": {"server": "backbone router"},
    "backbone router": {"server": "data-centre router"},
    "data-centre router": {"server": "server"},
}


def follow_route(tables, source, destination):
    """List of nodes from source to destination, found by asking each node for its next hop."""
    path = [source]
    while path[-1] != destination:
        nxt = tables[path[-1]][destination]
        if nxt in path:
            raise RuntimeError("routing loop at " + nxt)
        path.append(nxt)
    return path

## Step 3: how many time steps does the whole message need?
One *step* is the time one packet needs to cross one link. A link carries one packet at a
time, and a router can only pass on a packet after it has fully received it. So the packets
follow each other like cars on a road: the first packet needs `hops` steps, and each of the
other packets arrives one step after the previous one.

    steps = hops + packets - 1

In [ ]:
def journey_steps(n_packets, hops):
    """Simulate the journey step by step and return the steps until all packets have arrived."""
    queue = [0] * (hops + 1)          # queue[i] = packets waiting at node i (node 0 = sender)
    queue[0] = n_packets
    steps = 0
    while queue[hops] < n_packets:
        movers = [i for i in range(hops) if queue[i] > 0]   # each link moves one packet per step
        for i in movers:
            queue[i] -= 1
            queue[i + 1] += 1
        steps += 1
    return steps


def delivery_time_ms(message_bytes, size, hops):
    """Time for the whole message: steps times the time one packet needs to cross one link."""
    n = packet_count(message_bytes, size)
    step_seconds = (size + HEADER_BYTES) * 8 / LINK_RATE
    return 1000 * (hops + n - 1) * step_seconds

## Step 4: see it
Left: the six nodes of a journey with the packets of a 4-packet message, one picture per time
step. Right: the delivery time against packet size for a 1000 byte message over 5 hops. Tiny
packets waste time on labels and need many steps; huge packets make every step long.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    msg = bytes(range(256)) * 4 + bytes(range(232))     # 1256 bytes of test data
    pk = packetize(msg, 300)
    shuffled = pk[:]
    random.Random(1).shuffle(shuffled)
    print("packets:", [(i, len(d)) for i, d in pk], "-> rebuilt equals original:", reassemble(shuffled) == msg)
    route = follow_route(NEXT_HOP, "laptop", "server")
    print("route:", " -> ".join(route))
    print("4 packets over 3 hops need", journey_steps(4, 3), "steps")

    fig, axs = plt.subplots(2, 3, figsize=(10, 4.2), sharex=True, sharey=True)
    hops, n = 5, 4
    for ax, step in zip(axs.ravel(), range(6)):
        for i in range(hops + 1):
            ax.plot(i, 0, "s", color="tab:gray", ms=10)
        ax.set_title(f"after step {step}", fontsize=9)
        for k in range(n):
            pos = min(max(step - k, 0), hops)      # packet k leaves one step after packet k-1
            ax.plot(pos, 0.15 + 0.12 * k, "o", color="tab:blue")
        ax.set_ylim(-0.2, 0.8)
        ax.set_yticks([])
        ax.set_xticks(range(hops + 1))
    for ax in axs[1]:
        ax.set_xlabel("node number (0 = laptop, 5 = server)")
    fig.suptitle("Four packets move forward like cars on a road (dots = packets)")
    fig.tight_layout()
    fig.savefig(OUT / "packet_journey.png", dpi=120)

    sizes = np.arange(20, 1001, 10)
    times = [delivery_time_ms(1000, int(s), 5) for s in sizes]
    best = int(sizes[int(np.argmin(times))])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(sizes, times)
    ax2.plot(best, min(times), "ro")
    ax2.set_xlabel("payload size per packet (bytes)")
    ax2.set_ylabel("delivery time of the whole message (ms)")
    ax2.set_title(f"1000 byte message, 5 hops: best size about {best} bytes")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "time_vs_packet_size.png", dpi=120)
    print("best payload size", best, "bytes ->", round(min(times), 2), "ms")
    print("one big packet:", round(delivery_time_ms(1000, 1000, 5), 2), "ms;  20 byte packets:",
          round(delivery_time_ms(1000, 20, 5), 2), "ms")

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the message in `lesson.py` to the text of a short poem as bytes, cut it into 20 byte packets, shuffle them and rebuild it. Print whether the result equals the original.
2. **Stretch.** Add a router to `NEXT_HOP` so the route has 7 nodes, and predict `journey_steps(4, 6)` before you run it.
3. **Explain.** Explain why the best packet size would move to larger values if the label shrank to 10 bytes. Then check it with `delivery_time_ms` after setting `HEADER_BYTES = 10`.

## Assessment
- **A. Conceptual.** Why does every packet carry a sequence number? What goes wrong without it?
- **B. Numerical.** A 900 byte message is sent in packets of 250 bytes of payload over 4 hops at 1 Mbit/s, with a 40 byte label per packet. Find the number of packets, their sizes, the steps and the delivery time.
- **C. Graph and simulation interpretation.** In the graph of delivery time against payload size (1000 byte message, 5 hops), the time is 25.92 ms for 20 byte packets, 15.68 ms for 100 byte packets and 41.6 ms for one 1000 byte packet. Explain the shape of the curve.
- **D. Design and optimisation challenge.** Design: for the 1000 byte message over 5 hops, compare payloads of 50, 100, 200 and 500 bytes and choose one. What would make you choose a larger size in real life?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import random


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    msg = bytes(range(200))
    pk = L.packetize(msg, 64)
    checks.append(("200 bytes in packets of 64 make 4 packets of 64, 64, 64 and 8 bytes",
                   [len(d) for _, d in pk] == [64, 64, 64, 8]))
    shuffled = pk[:]
    random.Random(3).shuffle(shuffled)
    checks.append(("packets that arrive in a shuffled order are put back into the original message",
                   L.reassemble(shuffled) == msg))
    checks.append(("an empty message makes no packets and rebuilds to nothing",
                   L.packetize(b"", 10) == [] and L.reassemble([]) == b""))
    try:
        L.packetize(b"abc", 0)
        bad = False
    except ValueError:
        bad = True
    checks.append(("a packet size of 0 is refused", bad))
    checks.append(("an exact multiple needs no extra packet: 600 bytes in 300 byte packets is 2",
                   L.packet_count(600, 300) == 2 and L.packet_count(601, 300) == 3))
    route = L.follow_route(L.NEXT_HOP, "laptop", "server")
    checks.append(("the route has 6 nodes and so 5 hops", len(route) == 6 and route[0] == "laptop" and route[-1] == "server"))
    looped = {"a": {"z": "b"}, "b": {"z": "a"}}
    try:
        L.follow_route(looped, "a", "z")
        caught = False
    except RuntimeError:
        caught = True
    checks.append(("a routing loop is detected instead of running forever", caught))
    checks.append(("a single packet over 5 hops needs 5 steps", L.journey_steps(1, 5) == 5))
    checks.append(("hand calculation: 4 packets over 3 hops need 3 + 4 - 1 = 6 steps", L.journey_steps(4, 3) == 6))
    checks.append(("steps = hops + packets - 1 for every case from 1 to 12 packets and 1 to 8 hops",
                   all(L.journey_steps(n, h) == h + n - 1 for n in range(1, 13) for h in range(1, 9))))
    # worked example: 1000 bytes, 300 byte payloads, 3 hops, 40 byte label, 1 Mbit/s links
    n = L.packet_count(1000, 300)
    t = L.delivery_time_ms(1000, 300, 3)
    checks.append((f"worked example: 1000 bytes in 300 byte packets is 4 packets, 6 steps, 16.32 ms (got {n}, {t:.2f})",
                   n == 4 and L.journey_steps(n, 3) == 6 and abs(t - 16.32) < 1e-9))
    checks.append(("one 1000 byte packet over 5 hops takes 41.6 ms",
                   abs(L.delivery_time_ms(1000, 1000, 5) - 41.6) < 1e-9))
    checks.append(("100 byte packets: 10 packets, 14 steps, 15.68 ms over 5 hops",
                   L.packet_count(1000, 100) == 10 and abs(L.delivery_time_ms(1000, 100, 5) - 15.68) < 1e-9))
    checks.append(("20 byte packets are slower than 100 byte packets (25.92 ms against 15.68 ms)",
                   abs(L.delivery_time_ms(1000, 20, 5) - 25.92) < 1e-9
                   and L.delivery_time_ms(1000, 20, 5) > L.delivery_time_ms(1000, 100, 5)))
    best = min(range(10, 1001, 10), key=lambda s: L.delivery_time_ms(1000, s, 5))
    checks.append(("the best payload size in steps of 10 bytes for 1000 bytes over 5 hops is 100", best == 100))
    pk = L.packetize(bytes(900), 250)
    checks.append(("900 bytes in 250 byte payloads: 4 packets of 250, 250, 250 and 150 bytes, 7 steps over 4 hops, 16.24 ms",
                   [len(d) for _, d in pk] == [250, 250, 250, 150] and L.journey_steps(4, 4) == 7
                   and abs(L.delivery_time_ms(900, 250, 4) - 16.24) < 1e-9))
    got = [round(L.delivery_time_ms(1000, s, 5), 2) for s in (50, 100, 200, 500)]
    checks.append((f"1000 bytes over 5 hops: payloads 50, 100, 200, 500 take 17.28, 15.68, 17.28, 25.92 ms (got {got})",
                   got == [17.28, 15.68, 17.28, 25.92]))
    checks.append(("the app default (1000 bytes, 300 byte packets, 5 hops) takes 21.76 ms",
                   abs(L.delivery_time_ms(1000, 300, 5) - 21.76) < 1e-9))
    checks.append(("20 byte packets: 50 packets, 54 steps over 5 hops",
                   L.packet_count(1000, 20) == 50 and L.journey_steps(50, 5) == 54))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
